In [0]:
%sql

-- Master_Cleaned START: write one child row linked to current master run
WITH active_run AS (
  SELECT MasterLogId
  FROM workspace.logs.master_log
  WHERE JobName = 'Training_Job' AND Status = 'RUNNING'
  ORDER BY MasterLogId DESC
  LIMIT 1
)
INSERT INTO workspace.logs.child_log (MasterLogId, NotebookName, StartTime, Status)
SELECT MasterLogId, 'Master_Cleaned', current_timestamp(), 'RUNNING'
FROM active_run;


In [0]:

# Databricks notebook source
# Parameters (optional)
dbutils.widgets.text("run_date", "")
run_date = dbutils.widgets.get("run_date")


# List your cleaned notebooks explicitly in the desired order
cleaned_notebooks = [
  "/Workspace/Users/aaryansatere@gmail.com/Training/Notebook/Cleaned/Cleaned_Countries",  
  "/Workspace/Users/aaryansatere@gmail.com/Training/Notebook/Cleaned/Cleaned_Orders",
  "/Workspace/Users/aaryansatere@gmail.com/Training/Notebook/Cleaned/Cleaned_products",
  "/Workspace/Users/aaryansatere@gmail.com/Training/Notebook/Cleaned/Cleaned_Rawtable",
  "/Workspace/Users/aaryansatere@gmail.com/Training/Notebook/Cleaned/Cleaned_sales"
]

for nb in cleaned_notebooks:
    print(f"Running: {nb}")
    # Propagate parameters as needed
    dbutils.notebook.run(nb, timeout_seconds=0, arguments={"run_date": run_date})



In [0]:
%sql

-- Master_Cleaned SUCCESS: mark this notebook as successful
UPDATE workspace.logs.child_log
SET EndTime = current_timestamp(), Status = 'SUCCESS'
WHERE MasterLogId = (
        SELECT MasterLogId
        FROM workspace.logs.master_log
        WHERE JobName = 'Training_Job' AND Status = 'RUNNING'
        ORDER BY MasterLogId DESC
        LIMIT 1
     )
  AND NotebookName = 'Master_Cleaned'
  AND Status = 'RUNNING';
